In [ ]:
import os
import json
import torch
from transformers import AutoTokenizer
from transformers.models.qwen3 import modeling_qwen3
from transformers import BitsAndBytesConfig

from lxt.efficient import monkey_patch

monkey_patch(modeling_qwen3, verbose=True)

# ============== Configuration ==============
# Backprop mode options:
#   - "max_logit": backprop on the maximum logit (default)
#   - "logit_diff": backprop on the difference between top logit and a logit at a given rank
# BACKPROP_MODE = "max_logit"  
BACKPROP_MODE = "logit_diff"  
CONTRAST_RANK = 7  # rank of the logit to contrast with (2 = second highest, 3 = third, etc.)
# CONTRAST_RANK = 13  # rank of the logit to contrast with (2 = second highest, 3 = third, etc.)

# optional 4bit quantization 
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16, # use bfloat16 to prevent overflow in gradients
)


def hook_hidden_activation(module, input, output):
    if isinstance(output, tuple):
        output = output[0]

    # save the activation and make sure the gradient is also saved in the .grad attribute after the backward pass
    module.output = output
    module.output.retain_grad() if module.output.requires_grad else None

caiw_dir = "/mnt/caiw"
fp = r"exp/quick_test/traces/quick_test_1_prompt_idx_1.json" # CONTRAST_RANK = 13
# fp = r"exp/quick_test/traces/quick_test_1_prompt_idx_4.json"
# fp = r"exp/quick_test/traces/quick_test_2.json"
# fp = r"exp/quick_test/traces/quick_test_2_prompt_idx_2.json"
# fp = r"exp/quick_test/traces/quick_test_2_prompt_idx_2_model_2.json" # correct
# fp = r"exp/quick_test/traces/quick_test_3.json"

with open(os.path.join(caiw_dir, fp), "r") as f:
    sample = json.load(f)
prompt = f"{sample['query']} {sample['prediction_origin']}" if sample['query'] else sample['prediction_origin']
path = sample['metadata']["model"]
model_name = path.split('/')[-1]

# prompt = f"I have 5 cats and 3 dogs. My cats love to play with my"
# path = 'Qwen/Qwen3-0.6B'
# model_name = "qwen3_0.6b"

print(f"Loading model from {path}...")

model = modeling_qwen3.Qwen3ForCausalLM.from_pretrained(path, device_map='cuda', torch_dtype=torch.bfloat16, quantization_config=quantization_config)
tokenizer = AutoTokenizer.from_pretrained(path)

# optional gradient checkpointing to save memory (2x forward pass)
model.train()
model.gradient_checkpointing_enable()

# deactive gradients on parameters to save memory
for param in model.parameters():
    param.requires_grad = False

# apply hooks and store handles for later removal/re-registration
hook_handles = []
for layer in model.model.layers:
    handle = layer.register_forward_hook(hook_hidden_activation)
    hook_handles.append(handle)

# forward & backward pass
input_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).input_ids.to(model.device)
input_embeds = model.get_input_embeddings()(input_ids)

outputs = model(inputs_embeds=input_embeds.requires_grad_(), use_cache=False, output_hidden_states=True)
output_logits = outputs.logits
last_logits = output_logits[0, -1, :]

# Get sorted logits and indices
sorted_logits, sorted_indices = torch.sort(last_logits, dim=-1, descending=True)

if BACKPROP_MODE == "max_logit":
    # Backprop on the maximum logit
    target_logit = sorted_logits[0]
    target_token = tokenizer.convert_ids_to_tokens(sorted_indices[:, 0])
    print(f"Prediction (top-1): {target_token}")
    print(f"Top logit value: {sorted_logits[0].item():.4f}")
    mode_desc = "Max Logit"
    
elif BACKPROP_MODE == "logit_diff":
    # Backprop on the difference between top logit and logit at CONTRAST_RANK
    top_logit = sorted_logits[0]
    contrast_logit = sorted_logits[CONTRAST_RANK - 1]
    target_logit = top_logit - contrast_logit
    
    top_token = tokenizer.convert_ids_to_tokens([sorted_indices[0]])
    contrast_token = tokenizer.convert_ids_to_tokens([sorted_indices[CONTRAST_RANK - 1]])
    
    print(f"Top-1 prediction: {top_token} (logit: {top_logit.item():.4f})")
    print(f"Rank-{CONTRAST_RANK} prediction: {contrast_token} (logit: {contrast_logit.item():.4f})")
    print(f"Logit difference: {target_logit.item():.4f}")
    mode_desc = f"Logit Diff (Top-1 vs Rank-{CONTRAST_RANK})"
else:
    raise ValueError(f"Unknown BACKPROP_MODE: {BACKPROP_MODE}")

h = outputs.hidden_states[-1] 
h.retain_grad()

target_logit.backward()

## circuit formation

In [ ]:
# target_layer_idx, source_layer_idx = 27, 26
# target_layer_idx, source_layer_idx = 27, 24
# target_layer_idx, source_layer_idx = 25, 20
target_layer_idx, source_layer_idx = 20, 5

In [ ]:
target_layer = model.model.layers[target_layer_idx]
source_layer = model.model.layers[source_layer_idx]

relevance_target_layer = (target_layer.output * target_layer.output.grad)
relevance_source_layer = (source_layer.output * source_layer.output.grad)

print("original output:")
print(source_layer.output[0, -1, :20])
print(target_layer.output[0, -1, :20])

print("original gradients:")
print(source_layer.output.grad[0, -1, :20])
print(target_layer.output.grad[0, -1, :20])

print("original source relevance:")
print(relevance_source_layer[0, -1, :20])

print("original target relevance:")
print(relevance_target_layer[0, -1, :20])

### check grad and relevance reconstruction

In [ ]:
# Define the stack of layers to run: from source_layer_idx + 1 to target_layer_idx
layer_indices = range(source_layer_idx + 1, target_layer_idx + 1)
layers_to_run = [model.model.layers[i] for i in layer_indices]
print(f"Running layers: {list(layer_indices)}")

# Temporarily remove hooks for ALL layers in the stack to prevent overwriting
removed_hooks = []
for i in layer_indices:
    if i < len(hook_handles) and hook_handles[i] is not None:
        hook_handles[i].remove()
        removed_hooks.append(i)

# Temporarily disable gradient checkpointing BEFORE forward pass
was_checkpointing = model.is_gradient_checkpointing
if was_checkpointing:
    model.gradient_checkpointing_disable()

try:
    target_layer_input = source_layer.output.detach().requires_grad_(True)

    # Get position embeddings from the model's rotary_emb
    position_ids = torch.arange(0, target_layer_input.shape[1], dtype=torch.long, device=target_layer_input.device).unsqueeze(0)
    position_embeddings = model.model.rotary_emb(target_layer_input, position_ids)

    # Forward pass through the stack of layers
    hidden_states = target_layer_input
    for layer in layers_to_run:
        if isinstance(hidden_states, tuple):
             hidden_states = hidden_states[0]
             
        hidden_states = layer(
            hidden_states,
            position_embeddings=position_embeddings,
            use_cache=False
        )
        if isinstance(hidden_states, tuple):
             hidden_states = hidden_states[0]
             
    reconstructed_target_layer_output = hidden_states

    # ============ backward pass with original gradient ============
    custom_grad_output = target_layer.output.grad

    reconstructed_grad_input = torch.autograd.grad(
        outputs=reconstructed_target_layer_output,
        inputs=target_layer_input,
        grad_outputs=custom_grad_output,
        retain_graph=False  # IMPORTANT: keep graph for next backward pass
    )[0]

    reconstructed_relevance_source_layer = reconstructed_grad_input * target_layer_input

finally:
    # Re-enable gradient checkpointing if it was enabled
    if was_checkpointing:
        model.gradient_checkpointing_enable()
        
    # Re-register hooks
    for i in removed_hooks:
        hook_handles[i] = model.model.layers[i].register_forward_hook(hook_hidden_activation)

print("recomputed output:")
print(reconstructed_target_layer_output[0, -1, :20])

print("\noriginal source gradient (preserved because hook was disabled):")
print(source_layer.output.grad[0, -1, :20])

print("\n=== backward (with custom grad) ===")
print(relevance_target_layer[0, -1, :20])
print(reconstructed_relevance_source_layer[0, -1, :20])
print(relevance_source_layer.sum())
print(reconstructed_relevance_source_layer.sum())

### check relevance reconstruction using batch size; and diagonal along seq_len for batch_size dim

In [ ]:
# Define the stack of layers to run: from source_layer_idx + 1 to target_layer_idx
layer_indices = range(source_layer_idx + 1, target_layer_idx + 1)
layers_to_run = [model.model.layers[i] for i in layer_indices]
print(f"Running layers: {list(layer_indices)}")

# Temporarily remove hooks for ALL layers in the stack to prevent overwriting
removed_hooks = []
for i in layer_indices:
    if i < len(hook_handles) and hook_handles[i] is not None:
        hook_handles[i].remove()
        removed_hooks.append(i)

# Temporarily disable gradient checkpointing BEFORE forward pass
was_checkpointing = model.is_gradient_checkpointing
if was_checkpointing:
    model.gradient_checkpointing_disable()

try:
    target_layer_input = source_layer.output.detach()

    # Expand first dimension to match sequence length
    batch_size, seq_len, hidden_dim = target_layer_input.shape
    expanded_target_layer_input = target_layer_input.expand(seq_len, seq_len, hidden_dim).clone().requires_grad_(True)

    # Get position embeddings from the model's rotary_emb
    position_ids = torch.arange(0, expanded_target_layer_input.shape[1], dtype=torch.long, device=expanded_target_layer_input.device).unsqueeze(0)
    position_embeddings = model.model.rotary_emb(expanded_target_layer_input, position_ids)

    # Forward pass through the stack of layers
    hidden_states = expanded_target_layer_input
    for layer in layers_to_run:
        if isinstance(hidden_states, tuple):
             hidden_states = hidden_states[0]
             
        hidden_states = layer(
            hidden_states,
            position_embeddings=position_embeddings,
            use_cache=False
        )
        if isinstance(hidden_states, tuple):
             hidden_states = hidden_states[0]
             
    reconstructed_target_layer_output = hidden_states

    # ============ backward pass with different gradient ============
    epsilon = 1e-6
    
    expanded_target_layer_output_grad = torch.zeros(
        seq_len, seq_len, hidden_dim, 
        dtype=relevance_target_layer.dtype, 
        device=relevance_target_layer.device
    )
    # Use advanced indexing to set diagonal elements
    idx = torch.arange(seq_len, device=relevance_target_layer.device)
    expanded_target_layer_output_grad[idx, idx, :] = target_layer.output.grad[0, :, :]
    
    reconstructed_grad_input = torch.autograd.grad(
        outputs=reconstructed_target_layer_output,
        inputs=expanded_target_layer_input,
        grad_outputs=expanded_target_layer_output_grad,
        retain_graph=False  # IMPORTANT: keep graph for next backward pass
    )[0]

    reconstructed_relevance_source_layer = reconstructed_grad_input * target_layer_input

finally:
    # Re-enable gradient checkpointing if it was enabled
    if was_checkpointing:
        model.gradient_checkpointing_enable()
    
    # Re-register hooks
    for i in removed_hooks:
        hook_handles[i] = model.model.layers[i].register_forward_hook(hook_hidden_activation)

print(f"Expanded input shape: {expanded_target_layer_input.shape}")
print(f"Reconstructed output shape: {reconstructed_target_layer_output.shape}")
print(f"Reconstructed relevance shape: {reconstructed_relevance_source_layer.shape}")

print(f"\nReconstructed relevance sum: {reconstructed_relevance_source_layer.sum()}")
print(f"Original source relevance sum: {relevance_source_layer.sum()}")

In [ ]:
token_interconnection = reconstructed_relevance_source_layer.sum(dim=-1)
# token_interconnection = token_interconnection[1:, 1:]
token_interconnection.shape

In [ ]:
token_interconnection[:2, :2]

In [ ]:
import numpy as np

# Ensure we have the data
if 'reconstructed_relevance_source_layer' not in locals():
    raise ValueError("Please run the reconstruction cell first!")

# Sum over hidden dimension to get token-to-token relevance
# Shape: [seq_len, seq_len] where [target_token, source_token]
token_interconnection = reconstructed_relevance_source_layer.sum(dim=-1)

# Prepare data
matrix = token_interconnection.detach().cpu().float().numpy()
seq_len = matrix.shape[0]

# Get token labels
if 'input_ids' in locals():
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    token_labels = [t.replace('Ġ', ' ').replace('▁', ' ').strip() for t in tokens]
else:
    token_labels = [f"Token_{i}" for i in range(seq_len)]

# Normalize interconnection values for visualization
abs_max = np.abs(matrix).max()
matrix_normalized = matrix / abs_max if abs_max > 0 else matrix

# Calculate node relevances
# For target nodes (top row): sum over sources (row sum)
target_relevances = matrix.sum(axis=1).tolist()  # [seq_len]
# For source nodes (bottom row): sum over targets (column sum)  
source_relevances = matrix.sum(axis=0).tolist()  # [seq_len]

# Create edges data (only include significant connections to avoid clutter)
threshold = 0.001  # Only show edges with normalized strength > threshold
edges = []
for target_idx in range(seq_len):
    for source_idx in range(seq_len):
        strength = matrix_normalized[target_idx, source_idx]
        if abs(strength) > threshold:
            edges.append({
                'source': source_idx,
                'target': target_idx,
                'strength': float(strength),
                'abs_strength': float(abs(strength)),
                'original_value': float(matrix[target_idx, source_idx])
            })

# Calculate canvas width based on sequence length (give each token ~15px of space)
canvas_width = max(1500, seq_len * 15)
canvas_height = 400

# Generate HTML with embedded JavaScript
html_content = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Token Interconnection Network</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background-color: #f5f5f5;
        }}
        #container {{
            background: white;
            border-radius: 8px;
            padding: 20px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }}
        #info {{
            margin-bottom: 20px;
            padding: 15px;
            background: #e3f2fd;
            border-radius: 4px;
            border-left: 4px solid #2196F3;
        }}
        #controls {{
            margin-bottom: 20px;
            padding: 10px;
            background: #f5f5f5;
            border-radius: 4px;
        }}
        #canvas-wrapper {{
            position: relative;
            overflow-x: auto;
            overflow-y: hidden;
            border: 1px solid #ddd;
            border-radius: 4px;
        }}
        #canvas {{
            cursor: crosshair;
            display: block;
        }}
        .control-group {{
            margin: 10px 0;
        }}
        label {{
            display: inline-block;
            width: 180px;
            font-weight: bold;
        }}
        input[type="range"] {{
            width: 200px;
        }}
        #tooltip {{
            position: fixed;
            background: rgba(0, 0, 0, 0.85);
            color: white;
            padding: 10px;
            border-radius: 4px;
            font-size: 12px;
            pointer-events: none;
            opacity: 0;
            transition: opacity 0.2s;
            max-width: 300px;
            z-index: 1000;
        }}
        #legend {{
            margin-top: 20px;
            padding: 15px;
            background: #f9f9f9;
            border-radius: 4px;
        }}
        .legend-item {{
            display: inline-block;
            margin-right: 20px;
        }}
        .legend-color {{
            display: inline-block;
            width: 20px;
            height: 3px;
            margin-right: 5px;
            vertical-align: middle;
        }}
    </style>
</head>
<body>
    <div id="container">
        <h2>Token Interconnection Network (Layer {source_layer_idx} → {target_layer_idx})</h2>
        
        <div id="info">
            <strong>Instructions:</strong> Click on a node to highlight its connections. Hover over nodes and edges for details.
            <br><strong>Visualization:</strong> Top row = Target tokens, Bottom row = Source tokens
            <br><strong>Tip:</strong> Use horizontal scroll to navigate through all tokens
        </div>
        
        <div id="controls">
            <div class="control-group">
                <label>Edge Threshold:</label>
                <input type="range" id="threshold" min="0" max="100" value="10" step="1">
                <span id="threshold-value">0.10</span>
            </div>
            <div class="control-group">
                <label>Node Relevance Threshold:</label>
                <input type="range" id="relthreshold" min="0" max="100" value="10" step="1">
                <span id="relthreshold-value">0.10</span>
            </div>
            <div class="control-group">
                <label>Node Size:</label>
                <input type="range" id="nodesize" min="1" max="8" value="3" step="0.5">
                <span id="nodesize-value">3</span>
            </div>
            <div class="control-group">
                <label>Edge Width Scale:</label>
                <input type="range" id="edgewidth" min="1" max="10" value="3" step="0.5">
                <span id="edgewidth-value">3</span>
            </div>
        </div>
        
        <div id="canvas-wrapper">
            <canvas id="canvas" width="{canvas_width}" height="{canvas_height}"></canvas>
        </div>
        
        <div id="legend">
            <div class="legend-item">
                <span class="legend-color" style="background: rgba(255, 0, 0, 0.6);"></span>
                <span>Positive Relevance</span>
            </div>
            <div class="legend-item">
                <span class="legend-color" style="background: rgba(0, 0, 255, 0.6);"></span>
                <span>Negative Relevance</span>
            </div>
            <div class="legend-item">
                <span style="font-weight: bold;">●</span> Source Token (Bottom)
            </div>
            <div class="legend-item">
                <span style="font-weight: bold;">●</span> Target Token (Top)
            </div>
        </div>
    </div>
    
    <div id="tooltip"></div>
    
    <script>
        const canvas = document.getElementById('canvas');
        const canvasWrapper = document.getElementById('canvas-wrapper');
        const ctx = canvas.getContext('2d');
        const tooltip = document.getElementById('tooltip');
        
        // Data
        const tokens = {repr(token_labels)};
        const edges = {edges};
        const seqLen = {seq_len};
        const targetRelevances = {target_relevances};
        const sourceRelevances = {source_relevances};
        
        // Layout parameters
        const margin = {{ top: 50, right: 50, bottom: 50, left: 50 }};
        const width = canvas.width - margin.left - margin.right;
        const height = canvas.height - margin.top - margin.bottom;
        const rowSpacing = height * 0.6;
        const nodeSpacing = width / (seqLen + 1);
        
        // State
        let selectedNode = null;
        let hoveredNode = null;
        let threshold = 0.1;
        let relThreshold = 0.1;
        let nodeSize = 3;
        let edgeWidthScale = 3;
        
        // Calculate max absolute relevance for normalization
        const maxAbsRelevance = Math.max(
            Math.max(...targetRelevances.map(Math.abs)),
            Math.max(...sourceRelevances.map(Math.abs))
        );
        
        // Calculate node positions
        const sourceNodes = tokens.map((token, i) => ({{
            x: margin.left + (i + 1) * nodeSpacing,
            y: margin.top + rowSpacing,
            index: i,
            token: token,
            type: 'source',
            relevance: sourceRelevances[i]
        }}));
        
        const targetNodes = tokens.map((token, i) => ({{
            x: margin.left + (i + 1) * nodeSpacing,
            y: margin.top,
            index: i,
            token: token,
            type: 'target',
            relevance: targetRelevances[i]
        }}));
        
        const allNodes = [...sourceNodes, ...targetNodes];
        
        // Helper function to get mouse position accounting for scroll
        function getMousePosition(e) {{
            const wrapperRect = canvasWrapper.getBoundingClientRect();
            const scrollLeft = canvasWrapper.scrollLeft;
            const scrollTop = canvasWrapper.scrollTop;
            const x = e.clientX - wrapperRect.left + scrollLeft;
            const y = e.clientY - wrapperRect.top + scrollTop;
            return {{ x, y }};
        }}
        
        // Event listeners for controls
        document.getElementById('threshold').addEventListener('input', (e) => {{
            threshold = e.target.value / 100;
            document.getElementById('threshold-value').textContent = threshold.toFixed(2);
            draw();
        }});
        
        document.getElementById('relthreshold').addEventListener('input', (e) => {{
            relThreshold = e.target.value / 100;
            document.getElementById('relthreshold-value').textContent = relThreshold.toFixed(2);
            draw();
        }});
        
        document.getElementById('nodesize').addEventListener('input', (e) => {{
            nodeSize = parseFloat(e.target.value);
            document.getElementById('nodesize-value').textContent = nodeSize;
            draw();
        }});
        
        document.getElementById('edgewidth').addEventListener('input', (e) => {{
            edgeWidthScale = parseFloat(e.target.value);
            document.getElementById('edgewidth-value').textContent = edgeWidthScale;
            draw();
        }});
        
        // Mouse events
        canvas.addEventListener('mousemove', handleMouseMove);
        canvas.addEventListener('click', handleClick);
        canvas.addEventListener('mouseleave', () => {{
            hoveredNode = null;
            tooltip.style.opacity = '0';
            draw();
        }});
        
        function handleMouseMove(e) {{
            const {{ x, y }} = getMousePosition(e);
            
            let foundNode = null;
            for (const node of allNodes) {{
                const dx = x - node.x;
                const dy = y - node.y;
                if (Math.sqrt(dx*dx + dy*dy) < nodeSize + 2) {{
                    foundNode = node;
                    break;
                }}
            }}
            
            if (foundNode !== hoveredNode) {{
                hoveredNode = foundNode;
                draw();
            }}
            
            if (hoveredNode) {{
                tooltip.innerHTML = `
                    <strong>${{hoveredNode.type === 'source' ? 'Source' : 'Target'}} Token</strong><br>
                    Position: ${{hoveredNode.index}}<br>
                    Token: "${{hoveredNode.token}}"<br>
                    Relevance: ${{hoveredNode.relevance.toFixed(4)}}
                `;
                tooltip.style.left = (e.clientX + 10) + 'px';
                tooltip.style.top = (e.clientY + 10) + 'px';
                tooltip.style.opacity = '1';
            }} else {{
                tooltip.style.opacity = '0';
            }}
        }}
        
        function handleClick(e) {{
            const {{ x, y }} = getMousePosition(e);
            
            let foundNode = null;
            for (const node of allNodes) {{
                const dx = x - node.x;
                const dy = y - node.y;
                if (Math.sqrt(dx*dx + dy*dy) < nodeSize + 2) {{
                    foundNode = node;
                    break;
                }}
            }}
            
            if (foundNode) {{
                if (selectedNode && selectedNode.index === foundNode.index && selectedNode.type === foundNode.type) {{
                    selectedNode = null;
                }} else {{
                    selectedNode = foundNode;
                }}
                draw();
            }}
        }}
        
        function draw() {{
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            
            // Draw title
            ctx.font = 'bold 12px Arial';
            ctx.fillStyle = '#666';
            ctx.textAlign = 'center';
            ctx.fillText('Target Tokens', width / 2 + margin.left, 20);
            ctx.fillText('Source Tokens', width / 2 + margin.left, canvas.height - 20);
            
            // Filter edges
            const filteredEdges = edges.filter(e => e.abs_strength > threshold);
            
            // Determine which edges and nodes to highlight
            const edgesToHighlight = new Set();
            const connectedNodes = new Set();
            
            if (selectedNode) {{
                filteredEdges.forEach((edge, idx) => {{
                    if (selectedNode.type === 'source' && edge.source === selectedNode.index) {{
                        edgesToHighlight.add(idx);
                        connectedNodes.add(`target-${{edge.target}}`);
                    }} else if (selectedNode.type === 'target' && edge.target === selectedNode.index) {{
                        edgesToHighlight.add(idx);
                        connectedNodes.add(`source-${{edge.source}}`);
                    }}
                }});
            }}
            
            // Draw edges
            filteredEdges.forEach((edge, idx) => {{
                const sourceNode = sourceNodes[edge.source];
                const targetNode = targetNodes[edge.target];
                
                const isHighlighted = edgesToHighlight.has(idx);
                const opacity = isHighlighted ? 0.8 : (selectedNode ? 0.1 : edge.abs_strength * 0.6);
                const lineWidth = (isHighlighted ? 2 : 1) * edge.abs_strength * edgeWidthScale;
                
                const color = edge.strength > 0 ? 
                    `rgba(255, 0, 0, ${{opacity}})` : 
                    `rgba(0, 0, 255, ${{opacity}})`;
                
                ctx.strokeStyle = color;
                ctx.lineWidth = lineWidth;
                ctx.beginPath();
                ctx.moveTo(sourceNode.x, sourceNode.y);
                ctx.lineTo(targetNode.x, targetNode.y);
                ctx.stroke();
            }});
            
            // Draw nodes
            allNodes.forEach(node => {{
                const isSelected = selectedNode && selectedNode.index === node.index && selectedNode.type === node.type;
                const isHovered = hoveredNode && hoveredNode.index === node.index && hoveredNode.type === node.type;
                const isConnected = connectedNodes.has(`${{node.type}}-${{node.index}}`);
                
                // Node circle
                ctx.beginPath();
                ctx.arc(node.x, node.y, nodeSize, 0, 2 * Math.PI);
                
                if (isSelected) {{
                    ctx.fillStyle = '#FF9800';
                    ctx.strokeStyle = '#F57C00';
                    ctx.lineWidth = 3;
                }} else if (isHovered) {{
                    ctx.fillStyle = '#4CAF50';
                    ctx.strokeStyle = '#388E3C';
                    ctx.lineWidth = 2;
                }} else if (isConnected) {{
                    ctx.fillStyle = '#FFC107';
                    ctx.strokeStyle = '#FFA000';
                    ctx.lineWidth = 2;
                }} else {{
                    ctx.fillStyle = node.type === 'source' ? '#2196F3' : '#9C27B0';
                    ctx.strokeStyle = node.type === 'source' ? '#1976D2' : '#7B1FA2';
                    ctx.lineWidth = 1;
                }}
                
                ctx.fill();
                ctx.stroke();
                
                // Draw token labels (vertical text)
                ctx.save();
                ctx.font = 'bold 10px Arial';
                
                const labelOffset = nodeSize + 8;
                ctx.translate(node.x, node.y);
                ctx.rotate(-Math.PI / 2);
                
                let textX, textAlign;
                if (node.type === 'target') {{
                    textX = -labelOffset;
                    textAlign = 'right';
                }} else {{
                    textX = labelOffset;
                    textAlign = 'left';
                }}
                ctx.textAlign = textAlign;
                
                const textMetrics = ctx.measureText(node.token);
                const textWidth = textMetrics.width;
                const padding = 3;
                const rectHeight = 14;
                
                ctx.fillStyle = 'rgba(255, 255, 255, 0.95)';
                if (textAlign === 'right') {{
                    ctx.fillRect(textX - textWidth - padding, -rectHeight / 2, textWidth + padding * 2, rectHeight);
                }} else {{
                    ctx.fillRect(textX - padding, -rectHeight / 2, textWidth + padding * 2, rectHeight);
                }}
                
                if (isConnected) {{
                    ctx.fillStyle = '#D84315';
                }} else {{
                    ctx.fillStyle = '#666';
                }}
                ctx.fillText(node.token, textX, 3);
                
                ctx.restore();
                
                // Draw relevance value (vertical text) - only if above threshold or connected/selected/hovered
                const normalizedRel = Math.abs(node.relevance) / maxAbsRelevance;
                const showRelevance = normalizedRel > relThreshold || isConnected || isSelected || isHovered;
                
                if (showRelevance) {{
                    ctx.save();
                    ctx.font = (isConnected || isSelected) ? 'bold 9px Arial' : '9px Arial';
                    
                    const relValue = node.relevance.toFixed(2);
                    let relColor;
                    if (isConnected || isSelected) {{
                        relColor = node.relevance >= 0 ? '#B71C1C' : '#0D47A1';
                    }} else {{
                        relColor = node.relevance >= 0 ? '#C62828' : '#1565C0';
                    }}
                    
                    const relOffset = nodeSize + 55;
                    
                    ctx.translate(node.x, node.y);
                    ctx.rotate(-Math.PI / 2);
                    
                    let relTextX, relTextAlign;
                    if (node.type === 'target') {{
                        relTextX = -relOffset;
                        relTextAlign = 'right';
                    }} else {{
                        relTextX = relOffset;
                        relTextAlign = 'left';
                    }}
                    ctx.textAlign = relTextAlign;
                    
                    const relMetrics = ctx.measureText(relValue);
                    const relWidth = relMetrics.width;
                    const relPadding = 2;
                    const relRectHeight = 12;
                    
                    ctx.fillStyle = (isConnected || isSelected) ? 'rgba(255, 255, 200, 0.95)' : 'rgba(255, 255, 255, 0.9)';
                    if (relTextAlign === 'right') {{
                        ctx.fillRect(relTextX - relWidth - relPadding, -relRectHeight / 2, relWidth + relPadding * 2, relRectHeight);
                    }} else {{
                        ctx.fillRect(relTextX - relPadding, -relRectHeight / 2, relWidth + relPadding * 2, relRectHeight);
                    }}
                    
                    ctx.fillStyle = relColor;
                    ctx.fillText(relValue, relTextX, 3);
                    
                    ctx.restore();
                }}
            }});
        }}
        
        // Initial draw
        draw();
    </script>
</body>
</html>
"""

# Save to file
output_file = "token_interconnection_network.html"
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"Network visualization saved to {output_file}")
print(f"Canvas size: {canvas_width}x{canvas_height} pixels")
print(f"Total edges (before filtering): {len(edges)}")
print(f"Sequence length: {seq_len}")
print(f"Edge strength range: [{matrix.min():.4f}, {matrix.max():.4f}]")
print(f"\nOpen the HTML file in a browser to interact with the visualization.")
print(f"Use horizontal scrolling to navigate through all tokens.")

## check relevance distribution

In [ ]:
from matplotlib import pyplot as plt

for l in range(len(model.model.layers)-1, -1, -1):
    layer = model.model.layers[l]
    relevance = (layer.output[:, 1:, :] * layer.output.grad[:, 1:, :])

    plt.plot(relevance.sum(dim=-1)[0].detach().cpu().float())
    # plt.yscale('log')
    plt.ylabel('Relevance (log scale)')
    plt.xlabel('Token Position')
    plt.title(f'Relevance per Token (Log Scale) - Layer: {l}')
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# l_rel = relevance_last_layer[0, -1, :].detach().cpu().float().numpy()

l = 0
layer = model.model.layers[l]
l_rel = (layer.output[:, 1:, :] * layer.output.grad[:, 1:, :])
l_rel = l_rel.sum(dim=-1)[0].detach().cpu().float().numpy()

# l_rel

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

# Sort by absolute value (descending) to see most important elements first
sorted_indices = np.argsort(np.abs(l_rel))[::-1]
sorted_rel = l_rel[sorted_indices]

# Compute cumulative sum of absolute relevance
cumsum_abs = np.cumsum(np.abs(sorted_rel))
total_abs = np.abs(l_rel).sum()

# Normalize to get percentage explained
cumsum_pct = cumsum_abs / total_abs * 100

# Plot cumulative curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.arange(1, len(cumsum_pct) + 1), cumsum_pct, linewidth=2)
ax.axhline(y=50, color='blue', linestyle='--', alpha=0.7, label='50% threshold')
ax.axhline(y=75, color='green', linestyle='--', alpha=0.7, label='75% threshold')
ax.axhline(y=90, color='r', linestyle='--', alpha=0.7, label='90% threshold')
ax.axhline(y=95, color='orange', linestyle='--', alpha=0.7, label='95% threshold')

# Find how many elements needed for 50%, 75%, 90% and 95%
n_50 = np.searchsorted(cumsum_pct, 50) + 1
n_75 = np.searchsorted(cumsum_pct, 75) + 1
n_90 = np.searchsorted(cumsum_pct, 90) + 1
n_95 = np.searchsorted(cumsum_pct, 95) + 1

ax.axvline(x=n_50, color='blue', linestyle=':', alpha=0.5)
ax.axvline(x=n_75, color='green', linestyle=':', alpha=0.5)
ax.axvline(x=n_90, color='r', linestyle=':', alpha=0.5)
ax.axvline(x=n_95, color='orange', linestyle=':', alpha=0.5)

ax.set_xlabel('Number of Elements (sorted by |relevance|)')
ax.set_ylabel('Cumulative Relevance Explained (%)')
ax.set_title(f'Cumulative Relevance Curve\n50%: {n_50} ({n_50/len(l_rel)*100:.1f}%), 75%: {n_75} ({n_75/len(l_rel)*100:.1f}%), 90%: {n_90} ({n_90/len(l_rel)*100:.1f}%), 95%: {n_95} ({n_95/len(l_rel)*100:.1f}%)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, len(l_rel)])
ax.set_ylim([0, 105])

plt.tight_layout()
plt.show()

print(f"Total elements: {len(l_rel)}")
print(f"Elements for 50% relevance: {n_50} ({n_50/len(l_rel)*100:.2f}%)")
print(f"Elements for 75% relevance: {n_75} ({n_75/len(l_rel)*100:.2f}%)")
print(f"Elements for 90% relevance: {n_90} ({n_90/len(l_rel)*100:.2f}%)")
print(f"Elements for 95% relevance: {n_95} ({n_95/len(l_rel)*100:.2f}%)")

In [ ]:
from matplotlib import pyplot as plt

plt.plot(relevance_last_layer[0, -1, :].detach().cpu().float())

## check bias term

In [ ]:
for l in range(len(model.model.layers)-1, -1, -1):
    layer = model.model.layers[l]
    relevance = (layer.output * layer.output.grad).float().sum(-1).detach().cpu()
    print(f"layer {l}: {relevance[0, :].sum()}")

In [ ]:
for l in range(len(model.model.layers)-1, -1, -1):
    layer = model.model.layers[l]
    relevance = (layer.output[:, 1:, :] * layer.output.grad[:, 1:, :]).float().sum(-1).detach().cpu()
    print(f"layer {l}: {relevance[0, :].sum()}")

In [ ]:
layer.output.shape